In [1]:
import json
import psycopg2
from dateutil import parser as dateparser

In [2]:
# ---------------------------------------------------------
# DATABASE CONNECTION
# ---------------------------------------------------------
conn = psycopg2.connect(
    host="localhost",
    database="mates",
    user="postgres",
    password="1"
)
conn.autocommit = True
cur = conn.cursor()

In [3]:
def get_or_create_source(source_url):
    cur.execute("""
        INSERT INTO sources (source_url, source_name)
        VALUES (%s, %s)
        ON CONFLICT (source_url) DO UPDATE SET source_name = EXCLUDED.source_name
        RETURNING source_id;
    """, (source_url, source_url))
    return cur.fetchone()[0]

In [4]:
def get_or_create_category(category_name):
    if not category_name:
        return None

    cur.execute("""
        INSERT INTO categories (category_name)
        VALUES (%s)
        ON CONFLICT (category_name) DO UPDATE SET category_name = EXCLUDED.category_name
        RETURNING category_id;
    """, (category_name,))
    return cur.fetchone()[0]

In [5]:
def get_or_create_tag(tag_name):
    cur.execute("""
        INSERT INTO tags (tag_name)
        VALUES (%s)
        ON CONFLICT (tag_name) DO UPDATE SET tag_name = EXCLUDED.tag_name
        RETURNING tag_id;
    """, (tag_name,))
    return cur.fetchone()[0]

In [6]:
def insert_article(article_data, source_id, category_id):
    publication_date = dateparser.parse(article_data["publication_date"]).date()
    scrape_date = dateparser.parse(article_data["scrape_date"])

    cur.execute("""
        INSERT INTO articles (
            url, source_id, category_id,
            publication_date, scrape_date,
            title, content,
            word_count, sentence_count, character_count,
            category_confidence
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (url) DO UPDATE SET
            title = EXCLUDED.title,
            content = EXCLUDED.content,
            word_count = EXCLUDED.word_count,
            sentence_count = EXCLUDED.sentence_count,
            character_count = EXCLUDED.character_count,
            category_id = EXCLUDED.category_id,
            category_confidence = EXCLUDED.category_confidence,
            updated_at = CURRENT_TIMESTAMP
        RETURNING article_id;
    """, (
        article_data["url"],
        source_id,
        category_id,
        publication_date,
        scrape_date,
        article_data["title"],
        article_data["content"],
        article_data.get("word_count", 0),
        article_data.get("sentence_count", 0),
        article_data.get("character_count", 0),
        article_data.get("category_confidence", 0.0)
    ))

    return cur.fetchone()[0]

In [7]:
def insert_article_tags(article_id, tags):
    for tag_name in tags:
        tag_id = get_or_create_tag(tag_name)
        cur.execute("""
            INSERT INTO article_tags (article_id, tag_id)
            VALUES (%s, %s)
            ON CONFLICT (article_id, tag_id) DO NOTHING;
        """, (article_id, tag_id))

In [8]:
# ---------------------------------------------------------
# PROCESS A SINGLE ARTICLE (CORRECT FLOW)
# ---------------------------------------------------------
def process_article(article):
    print(f"→ Processing: {article['title'][:50]}")

    # 1. Insert / get source
    source_id = get_or_create_source(article["source"])

    # 2. Insert / get category
    category_id = get_or_create_category(article.get("primary_category"))

    # 3. Insert article
    article_id = insert_article(article, source_id, category_id)

    # 4. Insert tags
    tags = article.get("tags", [])
    insert_article_tags(article_id, tags)

    print(f"Article inserted with ID {article_id}")

In [9]:
# ---------------------------------------------------------
# MAIN ETL FUNCTION
# ---------------------------------------------------------
def run_etl(json_path):
    print("Loading JSON...")
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Found {len(data)} articles.")

    for article in data:
        try:
            process_article(article)
        except Exception as e:
            print("Error:", e)

    print("ETL Completed Successfully!")

In [10]:
if __name__ == "__main__":
    run_etl(r"D:\Menghour\MATES\scraping\notebook\khmer_dual_complete_clean.json")

Loading JSON...
Found 1297 articles.
→ Processing: ថៃនៅតែបន្តបាញ់ប្រហារលើកម្ពុជា ទោះបីឯកភាពជាគោលការណ៍
Article inserted with ID 7857
→ Processing: កម្ពុជាត្រៀមបញ្ជូនអ្នកជំនាញដោះមីនទៅជួយបណ្ដុះបណ្ដាល
Article inserted with ID 7858
→ Processing: អ្នកបាញ់សម្លាប់លោក លិម គិមយ៉ា សារភាព តែមិនប្រាប់ពី
Article inserted with ID 7859
→ Processing: លោកនាយករដ្ឋមន្រ្តីស្នើ ឱ្យក្រសួងយុត្តិធម៌ពិនិត្យឡើ
Article inserted with ID 7860
→ Processing: តុលាការកំពូលសម្រេចទម្លាក់បទចោទលើពលរដ្ឋស្រុកតំបែរ៤ន
Article inserted with ID 7861
→ Processing: របាយការណ៍ ក្រុម ច្បាប់ ធុរកិច្ច និង សិទ្ធិមនុស្ស៖ 
Article inserted with ID 7862
→ Processing: ពិធីបុណ្យអុំទូកចាប់ផ្តើមនៅថ្ងៃនេះ បន្ទាប់ពីខកខានបួ
Article inserted with ID 7863
→ Processing: សាលាឧទ្ធរណ៍ភ្នំពេញតម្កល់សាលក្រមពីបទញុះញង់លើ លោក ថា
Article inserted with ID 7864
→ Processing: រដ្ឋាភិបាលប្រកាសផ្តល់ប្រាក់ឧបត្ថម្ភដល់កម្មករដែលត្រ
Article inserted with ID 7865
→ Processing: បុណ្យសមុទ្រនឹងប្រារព្ធនៅខេត្តព្រះសីហនុនៅចុងសប្តាហ៍
Article inserted with ID 7866
→ Processing: